In [ ]:
%%capture

# Grab all the datastructures from analysis
%run ./analysis.ipynb

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from functools import reduce
from math import log10 as log
from matplotlib import ticker as mticker

import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
SMALL_SIZE = 12
MEDIUM_SIZE = 16
LARGE_SIZE = 20

# Set values for figure and font sizes

plt.rc('figure', figsize=[9,6], dpi=100)
plt.rc('axes', titlesize=LARGE_SIZE, labelsize=LARGE_SIZE)
plt.rc('xtick', labelsize=LARGE_SIZE)
plt.rc('ytick', labelsize=LARGE_SIZE)
plt.rc('legend', fontsize=MEDIUM_SIZE)
plt.rc('lines', linewidth=4)

# Ensure output directory exists
os.makedirs('output', exist_ok=True)

# Consistent labels
def get_label(name):
    if name == 'machine_strength':
        return 'Random'
    elif name == 'human_strength':
        return 'Alphabetic'
    elif name == 'chinese_strength':
        return 'Numeric'
    else:
        raise ValueError('Invalid column name')
        
def get_long_label(name):
    if name == 'machine_strength':
        return 'entirely random'
    elif name == 'human_strength':
        return 'alphabetic-first'
    elif name == 'chinese_strength':
        return 'numeric-first'
    else:
        raise ValueError('Invalid column name')

In [ ]:
# Add analysis columns
policy['min_length'] = policy['policy'].apply(lambda p: min([rule.min_length for rule in p.rules]))
policy['max_length'] = policy['policy'].apply(lambda p: max([rule.max_length if rule.max_length is not None else math.inf for rule in p.rules]))
policy['max_length_stats'] = policy['policy'].apply(lambda p: max([rule.max_length if rule.max_length is not None and not math.isinf(rule.max_length) else 10**6 for rule in p.rules])) 

# Basic distribution of strengths

In [ ]:
fig, ax = plt.subplots()

sns.kdeplot(data=policy['machine_strength'], log_scale=True, label=get_label('machine_strength'), ax=ax)
sns.kdeplot(data=policy['human_strength'], log_scale=True, label=get_label('human_strength'), ax=ax)
sns.kdeplot(data=policy['chinese_strength'], log_scale=True, label=get_label('chinese_strength'), ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend(fontsize=LARGE_SIZE)
fig.savefig("output/overall_strength_distribution.pdf", bbox_inches='tight', dpi=600)

In [ ]:
fig, ax = plt.subplots()

sns.ecdfplot(data=policy['machine_strength'], log_scale=True, label=get_label('machine_strength'), ax=ax)
sns.ecdfplot(data=policy['human_strength'], log_scale=True, label=get_label('human_strength'), ax=ax)
sns.ecdfplot(data=policy['chinese_strength'], log_scale=True, label=get_label('chinese_strength'), ax=ax)

plt.plot([10**6, 10**6], [1, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [1, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("CDF")

ax.legend(fontsize=LARGE_SIZE)
fig.savefig("output/overall_strength_cdf.pdf", bbox_inches='tight', dpi=600)

In [ ]:
fig, ax = plt.subplots()

df = policy[['machine_strength', 'human_strength', 'chinese_strength']].melt(var_name='method', value_name='PCP Strength')
df['method'] = df['method'].apply(get_label)
df['PCP Strength'] = df['PCP Strength'].apply(math.log10)

sns.violinplot(x="method", y="PCP Strength", data=df, inner=None, ax=ax)

#plt.plot([10**6, 10**6], [1, 0], color='black', linestyle='dashed')
#plt.plot([10**14, 10**14], [1, 0], color='black', linestyle='dashed')

plt.xlabel(None)

plt.ylabel("PCP Strength")
plt.ylim(bottom=1, top=22)
plt.yticks([i for i in range(0, 23, 2)])
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

fig.savefig("output/overall_strength_violin.pdf", bbox_inches='tight', dpi=600)

# Min and Max lengths

In [ ]:
policy['min_length'].value_counts()

In [ ]:
policy[policy['min_length'] < 6]['min_length'].count()

In [ ]:
fig, ax = plt.subplots()

sns.countplot(x=policy['min_length'], color='tab:green', ax=ax)

plt.xlabel("Minimum Length")
plt.ylabel("Count")

#ax.label()

fig.savefig("output/overall_min_length.pdf", bbox_inches='tight', dpi=600)

In [ ]:
policy['max_length'].value_counts().sort_index()

In [ ]:
policy[policy['max_length'] <= 16]['min_length'].count()

In [ ]:
fig, ax = plt.subplots()

sns.ecdfplot(x=policy['max_length'], color='tab:red', ax=ax)

plt.xlabel("Maximum Length")

plt.ylabel("CDF")

#ax.label()

fig.savefig("output/overall_max_length.pdf", bbox_inches='tight', dpi=600)

# Rules used

In [ ]:
properties = policy['policy'].apply(lambda p: [
    name
    for rule in p.rules
    for name, value in rule.__dict__.items()
    if value is not None and not name.startswith('_')
])
properties.apply(str).value_counts()

In [ ]:
names = set(np.concatenate(properties).tolist())

for name in names:
    print(f'{name}: {properties.apply(lambda list: name in list).sum()}')

In [ ]:
requirements = policy['policy'].apply(lambda p: [
    requirement
    for rule in p.rules
    if rule.require is not None
    for requirement in rule.require]
)
requirements.apply(set).apply(str).value_counts()

In [ ]:
names = set(np.concatenate(requirements).tolist())

for name in names:
    print(f'{name}: {requirements.apply(lambda list: name in list).sum()}')

In [ ]:
subset = policy['policy'].apply(lambda p: [
    rule.require_subset
    for rule in p.rules
    if rule.require_subset is not None
])
subset.apply(str).value_counts()

In [ ]:
max_consecutive = policy['policy'].apply(lambda p: [
    rule.max_consecutive
    for rule in p.rules
    if rule.max_consecutive is not None]
)
max_consecutive.apply(str).value_counts()

In [ ]:
policy['policy_exclusions'].value_counts()

# By Country

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in country_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/country_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by country')
fig.savefig("output/country_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('machine_strength ~ country', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['machine_strength'], groups=policy['country']))

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in country_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/country_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by country')
fig.savefig("output/country_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('human_strength ~ country', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['human_strength'], groups=policy['country']))

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in country_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.30, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.30, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.30)
plt.yticks(np.arange(0, .35, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/country_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by country')
fig.savefig("output/country_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('chinese_strength ~ country', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['human_strength'], groups=policy['country']))

In [ ]:
fig, ax = plt.subplots()

for name, test in country_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

plt.title('PCP minimum length by country')
fig.savefig("output/country_minimum_length.pdf", bbox_inches='tight', dpi=600)

ax.legend()
fig.savefig("output/country_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('min_length ~ country', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['country']))

In [ ]:
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/country_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by country')
fig.savefig("output/country_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('max_length_stats ~ country', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# By popularity

In [ ]:
# Machine strength
binned_by_popularity = pd.DataFrame({name: policy[test]['machine_strength'] for name, test in popularity_bins.items()})
melted_bin_by_popularity = binned_by_popularity.melt(var_name='Popularity', value_name='Strength').dropna()
melted_bin_by_popularity['Strength'] = melted_bin_by_popularity['Strength'].apply(log)
sns.violinplot(x="Popularity", y="Strength", data=melted_bin_by_popularity, cut=0.1)

In [ ]:
# Human strength
binned_by_popularity = pd.DataFrame({name: policy[test]['human_strength'] for name, test in popularity_bins.items()})
melted_bin_by_popularity = binned_by_popularity.melt(var_name='Popularity', value_name='Strength').dropna()
melted_bin_by_popularity['Strength'] = melted_bin_by_popularity['Strength'].apply(log)
sns.violinplot(x="Popularity", y="Strength", data=melted_bin_by_popularity, cut=0.1)

In [ ]:
# Chinese strength
binned_by_popularity = pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in popularity_bins.items()})
melted_bin_by_popularity = binned_by_popularity.melt(var_name='Popularity', value_name='Strength').dropna()
melted_bin_by_popularity['Strength'] = melted_bin_by_popularity['Strength'].apply(log)
sns.violinplot(x="Popularity", y="Strength", data=melted_bin_by_popularity, cut=0.1)

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/popularity_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by popularity')
fig.savefig("output/popularity_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/popularity_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by popularity')
fig.savefig("output/popularity_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/popularity_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by popularity')
fig.savefig("output/popularity_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['machine_strength'] for name, test in popularity_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['human_strength'] for name, test in popularity_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['chinese_strength'] for name, test in popularity_bins.items()])

In [ ]:
# Non-binned strength
fig, ax = plt.subplots()

data = pd.DataFrame({
    name: value.apply(log)
    for name, value in policy[~policy['global_rank'].apply(math.isinf)][['global_rank', 'machine_strength', 'human_strength', 'chinese_strength']].iteritems()
})

sns.regplot(data=data, y="machine_strength", x="global_rank", fit_reg=True, label=get_label('machine_strength'), ax=ax, line_kws={'linewidth':4})
sns.regplot(data=data, y="human_strength", x="global_rank", fit_reg=True, label=get_label('human_strength'), ax=ax, line_kws={'linewidth':4})
sns.regplot(data=data, y="chinese_strength", x="global_rank", fit_reg=True, label=get_label('chinese_strength'), ax=ax, line_kws={'linewidth':4})

plt.plot([0, 7], [6,6], color='black', linestyle='dashed')
plt.plot([0, 7], [14,14], color='black', linestyle='dashed')

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("PCP Strength")
plt.ylim(bottom=1, top=20)
plt.yticks([i for i in range(0, 21, 2)])
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

ax.legend()
fig.savefig("output/rank_strength.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP strength by Alexa global rank')
fig.savefig("output/rank_strength_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Non-binned strength, machine strength
fig, ax = plt.subplots()

sns.regplot(data=data, y="machine_strength", x="global_rank", fit_reg=True, color='tab:blue', ax=ax, line_kws={'linewidth':4})

plt.plot([0, 7], [6,6], color='black', linestyle='dashed')
plt.plot([0, 7], [14,14], color='black', linestyle='dashed')

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("PCP Strength")
plt.ylim(bottom=1, top=20)
plt.yticks([i for i in range(0, 21, 2)])
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

fig.savefig("output/rank_machine_strength.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by Alexa global rank')
fig.savefig("output/rank_machine_strength_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Non-binned strength, human strength
fig, ax = plt.subplots()

sns.regplot(data=data, y="human_strength", x="global_rank", fit_reg=True, color='tab:orange', ax=ax, line_kws={'linewidth':4})

plt.plot([0, 7], [6,6], color='black', linestyle='dashed')
plt.plot([0, 7], [14,14], color='black', linestyle='dashed')

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("PCP Strength")
plt.ylim(bottom=1, top=20)
plt.yticks([i for i in range(0, 21, 2)])
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

fig.savefig("output/rank_human_strength.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by Alexa global rank')
fig.savefig("output/rank_human_strength_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Non-binned strength, chinese strength
fig, ax = plt.subplots()

sns.regplot(data=data, y="chinese_strength", x="global_rank", fit_reg=True, color='tab:green', ax=ax, line_kws={'linewidth':4})

plt.plot([0, 7], [6,6], color='black', linestyle='dashed')
plt.plot([0, 7], [14,14], color='black', linestyle='dashed')

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("PCP Strength")
plt.ylim(bottom=1, top=20)
plt.yticks([i for i in range(0, 21, 2)])
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

fig.savefig("output/rank_chinese_strength.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by Alexa global rank')
fig.savefig("output/rank_chinese_strength_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.pearsonr(data['global_rank'], data['machine_strength'])

In [ ]:
stats.pearsonr(data['global_rank'], data['human_strength'])

In [ ]:
stats.pearsonr(data['global_rank'], data['chinese_strength'])

In [ ]:
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/popularity_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by popularity')
fig.savefig("output/popularity_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Non-binned strength
fig, ax = plt.subplots()

data = pd.DataFrame({
    name: value
    for name, value in policy[~policy['global_rank'].apply(math.isinf)][['global_rank', 'min_length']].iteritems()
})
data['global_rank'] = data['global_rank'].apply(log)

sns.regplot(data=data, y="min_length", x="global_rank", fit_reg=True, color='tab:green', label="Machine", ax=ax, line_kws={'linewidth':4})

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("Minimum Length")

fig.savefig("output/rank_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by Alexa global rank')
fig.savefig("output/rank_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.pearsonr(data['global_rank'], data['min_length'])

In [ ]:
stats.f_oneway(*[policy[test]['min_length'] for name, test in popularity_bins.items()])

In [ ]:
lm=ols('min_length ~ popularity', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['popularity']))

In [ ]:
fig, ax = plt.subplots()

for name, test in popularity_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/popularity_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by popularity')
fig.savefig("output/popularity_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Non-binned strength
fig, ax = plt.subplots()

data = pd.DataFrame({
    name: value
    for name, value in policy[~policy['global_rank'].apply(math.isinf)][['global_rank', 'max_length', 'max_length_stats']].iteritems()
})
data['global_rank'] = data['global_rank'].apply(log)

sns.regplot(data=data, y="max_length", x="global_rank", fit_reg=False, color='tab:red', label="Machine", ax=ax, line_kws={'linewidth':4})

plt.xlabel("Alexa Global Rank")
plt.xlim(left=0, right=7)
plt.xticks([i for i in range(0, 8, 1)])
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

plt.ylabel("Maximum Length")
#plt.ylim(bottom=1, top=15)
#plt.yticks([i for i in range(0, 16, 1)])

fig.savefig("output/rank_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by Alexa global rank')
fig.savefig("output/rank_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.pearsonr(data['global_rank'], data['max_length_stats'])

In [ ]:
lm=ols('max_length_stats ~ popularity', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# By category

In [ ]:
# Machine strength
binned_by_category = pd.DataFrame({name: policy[test]['machine_strength'] for name, test in category_bins.items()})
melted_bin_by_category = binned_by_category.melt(var_name='Category', value_name='Strength').dropna()
melted_bin_by_category['Strength'] = melted_bin_by_category['Strength'].apply(log)
sns.violinplot(x="Category", y="Strength", data=melted_bin_by_category, cut=0.1)

In [ ]:
# Human strength
binned_by_category = pd.DataFrame({name: policy[test]['human_strength'] for name, test in category_bins.items()})
melted_bin_by_category = binned_by_category.melt(var_name='Category', value_name='Strength').dropna()
melted_bin_by_category['Strength'] = melted_bin_by_category['Strength'].apply(log)
sns.violinplot(x="Category", y="Strength", data=melted_bin_by_category, cut=0.1)

In [ ]:
# Chinese strength
binned_by_category = pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in category_bins.items()})
melted_bin_by_category = binned_by_category.melt(var_name='Category', value_name='Strength').dropna()
melted_bin_by_category['Strength'] = melted_bin_by_category['Strength'].apply(log)
sns.violinplot(x="Category", y="Strength", data=melted_bin_by_category, cut=0.1)

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in category_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/category_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by use case')
fig.savefig("output/category_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in category_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/category_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by use case')
fig.savefig("output/category_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in category_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.30, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.30, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.30)
plt.yticks(np.arange(0, .35, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/category_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by use case')
fig.savefig("output/category_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['machine_strength'] for name, test in category_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['human_strength'] for name, test in category_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['chinese_strength'] for name, test in category_bins.items()])

In [ ]:
fig, ax = plt.subplots()


for name, test in category_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/category_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by use case')
fig.savefig("output/category_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['min_length'] for name, test in category_bins.items()])

In [ ]:
lm=ols('min_length ~ use_case', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['use_case']))

In [ ]:
fig, ax = plt.subplots()

for name, test in category_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/category_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by use case')
fig.savefig("output/category_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('max_length_stats ~ use_case', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# By ads

In [ ]:
ads_bins = {'Yes': policy['ads'], 'No': ~policy['ads']}

In [ ]:
# Machine strength
binned_by_ads = pd.DataFrame({name: policy[test]['machine_strength'] for name, test in ads_bins.items()})
melted_bin_by_ads = binned_by_ads.melt(var_name='Ad provider', value_name='Strength').dropna()
melted_bin_by_ads['Strength'] = melted_bin_by_ads['Strength'].apply(log)
sns.violinplot(x="Ad provider", y="Strength", data=melted_bin_by_ads, cut=0.1)

In [ ]:
# Human strength
binned_by_ads = pd.DataFrame({name: policy[test]['human_strength'] for name, test in ads_bins.items()})
melted_bin_by_ads = binned_by_ads.melt(var_name='Ad provider', value_name='Strength').dropna()
melted_bin_by_ads['Strength'] = melted_bin_by_ads['Strength'].apply(log)
sns.violinplot(x="Ad provider", y="Strength", data=melted_bin_by_ads, cut=0.1)

In [ ]:
# Chinese strength
binned_by_ads = pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in ads_bins.items()})
melted_bin_by_ads = binned_by_ads.melt(var_name='Ad provider', value_name='Strength').dropna()
melted_bin_by_ads['Strength'] = melted_bin_by_ads['Strength'].apply(log)
sns.violinplot(x="Ad provider", y="Strength", data=melted_bin_by_ads, cut=0.1)

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in ads_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/ads_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by ad provider')
fig.savefig("output/ads_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in ads_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/ads_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by ad provider')
fig.savefig("output/ads_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in ads_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.30, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.30, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.30)
plt.yticks(np.arange(0, .35, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/ads_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by ad provider')
fig.savefig("output/ads_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['machine_strength'] for name, test in ads_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['human_strength'] for name, test in ads_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['chinese_strength'] for name, test in ads_bins.items()])

In [ ]:
fig, ax = plt.subplots()

for name, test in ads_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/ads_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by ad provider')
fig.savefig("output/ads_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['min_length'] for name, test in ads_bins.items()])

In [ ]:
lm=ols('min_length ~ ads', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['ads']))

In [ ]:
fig, ax = plt.subplots()

for name, test in ads_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/ads_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by ad provider')
fig.savefig("output/ads_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('max_length_stats ~ ads', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# By public usernames

In [ ]:
public_username_bins = {'Yes': policy['public_username'], 'No': ~policy['public_username']}

In [ ]:
# Machine strength
binned_by_public_username = pd.DataFrame({name: policy[test]['machine_strength'] for name, test in public_username_bins.items()})
melted_bin_by_public_username = binned_by_public_username.melt(var_name='Public usernames', value_name='Strength').dropna()
melted_bin_by_public_username['Strength'] = melted_bin_by_public_username['Strength'].apply(log)
sns.violinplot(x="Public usernames", y="Strength", data=melted_bin_by_public_username, cut=0.1)

In [ ]:
# Human strength
binned_by_public_username = pd.DataFrame({name: policy[test]['human_strength'] for name, test in public_username_bins.items()})
melted_bin_by_public_username = binned_by_public_username.melt(var_name='Public usernames', value_name='Strength').dropna()
melted_bin_by_public_username['Strength'] = melted_bin_by_public_username['Strength'].apply(log)
sns.violinplot(x="Public usernames", y="Strength", data=melted_bin_by_public_username, cut=0.1)

In [ ]:
# Chinese strength
binned_by_public_username = pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in public_username_bins.items()})
melted_bin_by_public_username = binned_by_public_username.melt(var_name='Public usernames', value_name='Strength').dropna()
melted_bin_by_public_username['Strength'] = melted_bin_by_public_username['Strength'].apply(log)
sns.violinplot(x="Public usernames", y="Strength", data=melted_bin_by_public_username, cut=0.1)

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in public_username_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/public_username_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by public usernames')
fig.savefig("output/public_username_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in public_username_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/public_username_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by public usernames')
fig.savefig("output/public_username_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in public_username_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.30, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.30, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.30)
plt.yticks(np.arange(0, .35, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/public_username_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by public usernames')
fig.savefig("output/public_username_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['machine_strength'] for name, test in public_username_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['human_strength'] for name, test in public_username_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['chinese_strength'] for name, test in public_username_bins.items()])

In [ ]:
fig, ax = plt.subplots()

for name, test in public_username_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/public_username_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by public usernames')
fig.savefig("output/public_username_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['min_length'] for name, test in public_username_bins.items()])

In [ ]:
lm=ols('min_length ~ public_username', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['public_username']))

In [ ]:
fig, ax = plt.subplots()

for name, test in public_username_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/public_username_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by public usernames')
fig.savefig("output/public_username_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('max_length_stats ~ public_username', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# By breaches

In [ ]:
data_breach_bins = {'Yes': policy['data_breach'], 'No': ~policy['data_breach']}

In [ ]:
# Machine strength
binned_by_data_breach = pd.DataFrame({name: policy[test]['machine_strength'] for name, test in data_breach_bins.items()})
melted_bin_by_data_breach = binned_by_data_breach.melt(var_name='Data breach', value_name='Strength').dropna()
melted_bin_by_data_breach['Strength'] = melted_bin_by_data_breach['Strength'].apply(log)
sns.violinplot(x="Data breach", y="Strength", data=melted_bin_by_data_breach, cut=0.1)

In [ ]:
# Human strength
binned_by_data_breach = pd.DataFrame({name: policy[test]['human_strength'] for name, test in data_breach_bins.items()})
melted_bin_by_data_breach = binned_by_data_breach.melt(var_name='Data breach', value_name='Strength').dropna()
melted_bin_by_data_breach['Strength'] = melted_bin_by_data_breach['Strength'].apply(log)
sns.violinplot(x="Data breach", y="Strength", data=melted_bin_by_data_breach, cut=0.1)

In [ ]:
# Chinese strength
binned_by_data_breach = pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in data_breach_bins.items()})
melted_bin_by_data_breach = binned_by_data_breach.melt(var_name='Data breach', value_name='Strength').dropna()
melted_bin_by_data_breach['Strength'] = melted_bin_by_data_breach['Strength'].apply(log)
sns.violinplot(x="Data breach", y="Strength", data=melted_bin_by_data_breach, cut=0.1)

In [ ]:
# Machine strength
fig, ax = plt.subplots()

for name, test in data_breach_bins.items():
    sns.kdeplot(data=policy[test]['machine_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/data_breach_strength_machine.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("machine_strength")} strength by data breach')
fig.savefig("output/data_breach_strength_machine_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Human strength
fig, ax = plt.subplots()

for name, test in data_breach_bins.items():
    sns.kdeplot(data=policy[test]['human_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.25, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.25, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.25)
plt.yticks(np.arange(0, .30, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/data_breach_strength_human.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("human_strength")} strength by data breach')
fig.savefig("output/data_breach_strength_human_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
# Chinese strength
fig, ax = plt.subplots()

for name, test in data_breach_bins.items():
    sns.kdeplot(data=policy[test]['chinese_strength'], log_scale=True, label=name, ax=ax)

plt.plot([10**6, 10**6], [0.30, 0], color='black', linestyle='dashed')
plt.plot([10**14, 10**14], [0.30, 0], color='black', linestyle='dashed')

plt.xlabel("PCP Strength")
plt.xlim(left=1, right=float(10**20))
plt.xticks([float(10**i) for i in range(0, 21, 2)])

plt.ylabel("Percentage")
plt.ylim(top=.30)
plt.yticks(np.arange(0, .35, .05))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
    
ax.legend()
fig.savefig("output/data_breach_strength_chinese.pdf", bbox_inches='tight', dpi=600)

plt.title(f'PCP {get_long_label("chinese_strength")} strength by data breach')
fig.savefig("output/data_breach_strength_chinese_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['machine_strength'] for name, test in data_breach_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['human_strength'] for name, test in data_breach_bins.items()])

In [ ]:
stats.f_oneway(*[policy[test]['chinese_strength'] for name, test in data_breach_bins.items()])

In [ ]:
fig, ax = plt.subplots()

for name, test in data_breach_bins.items():
    sns.kdeplot(data=policy[test]['min_length'], label=name, ax=ax)

plt.xlabel("Minimum Length")
plt.xlim(left=1, right=15)
plt.xticks(np.arange(1, 16, 1))

plt.ylabel("Percentage")
plt.ylim(top=.50)
plt.yticks(np.arange(0, .55, .10))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))

ax.legend()
fig.savefig("output/data_breach_minimum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP minimum length by data breach')
fig.savefig("output/data_breach_minimum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
stats.f_oneway(*[policy[test]['min_length'] for name, test in data_breach_bins.items()])

In [ ]:
lm=ols('min_length ~ data_breach', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

In [ ]:
print(pairwise_tukeyhsd(endog=policy['min_length'], groups=policy['data_breach']))

In [ ]:
fig, ax = plt.subplots()

for name, test in data_breach_bins.items():
    sns.ecdfplot(data=policy[test]['max_length'], label=name, ax=ax)

plt.xlabel("Maximum Length")
plt.ylabel("Percentage")

ax.legend()
fig.savefig("output/data_breach_maximum_length.pdf", bbox_inches='tight', dpi=600)

plt.title('PCP maximum length by data breach')
fig.savefig("output/data_breach_maximum_length_title.pdf", bbox_inches='tight', dpi=600)

In [ ]:
lm=ols('max_length_stats ~ data_breach', data=policy).fit()
sm.stats.anova_lm(lm, typ=2)

# Website Analysis

In [ ]:
print(f'Uses HTTP: {len(policy[~policy["https"]])}\n\n')
print(f'\n{policy[~policy["https"]]["country"].value_counts()}\n\n')
print(f'\n{policy[~policy["https"]]["popularity"].value_counts()}\n\n')
print(f'\n{policy[~policy["https"]]["use_case"].value_counts()}\n\n')

In [ ]:
print(policy[~policy['https']].sort_values('global_rank')[['country', 'popularity', 'use_case']].to_latex())

In [ ]:
print(f'Uses SSO: {len(policy[~pd.isna(policy["sso"])])}\n\n')
print(f'{policy[~pd.isna(policy["sso"])]["country"].value_counts()}\n\n')
print(f'{policy[~pd.isna(policy["sso"])]["popularity"].value_counts()}\n\n')
print(f'{policy[~pd.isna(policy["sso"])]["use_case"].value_counts()}\n\n')

In [ ]:
sso = policy[~pd.isna(policy["sso"])]['sso'].apply(list)
providers = set(np.concatenate(sso))
for name in sorted(providers):
    print(f'{name}: {sso.apply(lambda list: name in list).sum()}')

In [ ]:
len(providers)

In [ ]:
print(f'Uses strength meter: {len(policy[policy["strength_meter"]])}\n\n')
print(f'{policy[policy["strength_meter"]]["country"].value_counts()}\n\n')
print(f'{policy[policy["strength_meter"]]["popularity"].value_counts()}\n\n')
print(f'{policy[policy["strength_meter"]]["use_case"].value_counts()}\n\n')

In [ ]:
print(f'Uses strength check: {len(policy[policy["policy_strength_check"]])}\n\n')
print(f'{policy[policy["policy_strength_check"]]["country"].value_counts()}\n\n')
print(f'{policy[policy["policy_strength_check"]]["popularity"].value_counts()}\n\n')
print(f'{policy[policy["policy_strength_check"]]["use_case"].value_counts()}\n\n')

In [ ]:
print(f'Uses ads: {len(policy[policy["ads"]])}\n\n')
print(f'{policy[policy["ads"]]["country"].value_counts()}\n\n')
print(f'{policy[policy["ads"]]["popularity"].value_counts()}\n\n')
print(f'{policy[policy["ads"]]["use_case"].value_counts()}\n\n')

In [ ]:
print(f'Has public usernames: {len(policy[policy["public_username"]])}\n\n')
print(f'{policy[policy["public_username"]]["country"].value_counts()}\n\n')
print(f'{policy[policy["public_username"]]["popularity"].value_counts()}\n\n')
print(f'{policy[policy["public_username"]]["use_case"].value_counts()}\n\n')

In [ ]:
print(f'Has had a data breach: {len(policy[policy["data_breach"]])}\n\n')
print(f'{policy[policy["data_breach"]]["country"].value_counts()}\n\n')
print(f'{policy[policy["data_breach"]]["popularity"].value_counts()}\n\n')
print(f'{policy[policy["data_breach"]]["use_case"].value_counts()}\n\n')

# NIST Compliance

In [ ]:
policy[policy['min_length'] >= 8].count()

In [ ]:
policy[policy['max_length'] < math.inf].count()

In [ ]:
policy[policy['policy'].apply(PCP.dumps).str.contains('charset')].count()

In [ ]:
properties = policy['policy'].apply(lambda p: [
    name
    for rule in p.rules
    for name, value in rule.__dict__.items()
    if value is not None and not name.startswith('_')
])
properties.apply(str).value_counts()